In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-12b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [4]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [5]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="/content/drive/Shareddrives/Hackrare2026/gemma3_training.jsonl", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [16]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [18]:
dataset[100]

{'text': '<start_of_turn>user\nWhat are some examples of rare pediatric neurological conditions, and what are the key challenges in diagnosing and treating these conditions?<end_of_turn>\n<start_of_turn>model\nClinical Reasoning:\nMy approach begins with defining the scope of the question, identifying the key components: rare pediatric neurological conditions, diagnosis, and treatment.\n\nInitially, I need to examine what constitutes a "rare" neurological condition in pediatrics. I guidelines indicate consider that the definition can vary, but generally, it refers to conditions affecting a small number of individuals relative to the general population.\n\nFirst, I guidelines indicate consider examples of rare pediatric neurological conditions. To tackle this effectively, I looked for well-defined conditions with significant neurological involvement and characterized by their rarity. I included examples from different categories (lysosomal storage disorders, mitochondrial disorders, leu

In [6]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [7]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=16):   0%|          | 0/293 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/293 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/293 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 293 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 65,470,464 of 12,252,795,504 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.271600
2,1.250800
3,1.356400
4,1.141600
5,1.261700
6,0.997500
7,1.155100
8,0.975700
9,1.020600
10,1.067000


In [9]:
# Export to GGUF
model.save_pretrained_gguf(
    "modified_neuro_gemma3_12b",
    tokenizer,
    quantization_method = "q5_k_m",
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Unsloth: Merge process complete. Saved to `/content/modified_neuro_gemma3_12b`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.BF16.gguf', 'modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q5_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.Q5_K_M.gguf', 'modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.BF16-mmproj.gguf']


Unsloth: example usage for Multimodal LLMs: llama.cpp/llama-mtmd-cli -m modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.Q5_K_M.gguf --mmproj modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.BF16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
Unsloth: Saved Ollama Modelfile to modified_neuro_gemma3_12b_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f modified_neuro_gemma3_12b_gguf/Modelfile


{'save_directory': 'modified_neuro_gemma3_12b',
 'gguf_directory': 'modified_neuro_gemma3_12b_gguf',
 'gguf_files': ['modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.Q5_K_M.gguf',
  'modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.BF16-mmproj.gguf'],
 'modelfile_location': 'modified_neuro_gemma3_12b_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': True,
 'fix_bos_token': True}

In [23]:
from google.colab import files

# Download the GGUF model file
files.download("neuro_gemma3_12b_gguf/gemma-3-12b-it.Q8_0.gguf")

# Download the Modelfile
files.download("neuro_gemma3_12b_gguf/Modelfile")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from google.colab import drive
drive.mount('/content/drive')

!cp neuro_gemma3_12b_gguf/gemma-3-12b-it.Q8_0.gguf /content/drive/MyDrive/
!cp neuro_gemma3_12b_gguf/Modelfile /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub import login

login()

api = HfApi()
api.create_repo("slplayford/modified-neuro-gemma3-12b", exist_ok=True)

api.upload_file(
    path_or_fileobj="modified_neuro_gemma3_12b_gguf/gemma-3-12b-it.Q5_K_M.gguf",
    path_in_repo="gemma-3-12b-it.Q5_K_M.gguf",
    repo_id="slplayford/modified-neuro-gemma3-12b",
)

api.upload_file(
    path_or_fileobj="modified_neuro_gemma3_12b_gguf/Modelfile",
    path_in_repo="Modelfile",
    repo_id="slplayford/modified-neuro-gemma3-12b",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...emma-3-12b-it.Q5_K_M.gguf:   0%|          |  543kB / 8.45GB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/slplayford/neuro-gemma3-12b/commit/d4b7fa3373bcb3b6a816190b5e231acc4b26c01f', commit_message='Upload Modelfile with huggingface_hub', commit_description='', oid='d4b7fa3373bcb3b6a816190b5e231acc4b26c01f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/slplayford/neuro-gemma3-12b', endpoint='https://huggingface.co', repo_type='model', repo_id='slplayford/neuro-gemma3-12b'), pr_revision=None, pr_num=None)